In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN

import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage

In [ ]:
# ================================
# 1. LEITURA E PRÉ-PROCESSAMENTO
# ================================

# Ajuste o separador se necessário (ex: sep=';')
df = pd.read_csv('https://raw.githubusercontent.com/Sandrocamargo/data-science/refs/heads/main/datasets/CID-RS.txt', sep=';', header=0)

# Define primeira coluna como índice (doenças)
df.set_index(df.columns[0], inplace=True)

# Remove última linha e última coluna (totais)
df = df.iloc[:-1, :-1]

# Converte para numérico (caso tenha strings)
df = df.apply(pd.to_numeric, errors='coerce')

# Remove possíveis NaNs
df = df.dropna()

print("Shape final:", df.shape)
df.head()

In [ ]:
# ================================
# 2. NORMALIZAÇÃO
# ================================

scaler = StandardScaler()

# Para clustering de doenças (linhas)
X_diseases = scaler.fit_transform(df.values)

# Para clustering de anos (colunas)
X_years = scaler.fit_transform(df.T.values)

In [ ]:
# ================================
# 3. CLUSTERING - DOENÇAS
# ================================

print("\n=== CLUSTERING DE DOENÇAS ===")

# KMeans
kmeans_d = KMeans(n_clusters=3, random_state=42)
labels_kmeans_d = kmeans_d.fit_predict(X_diseases)

# Hierarchical
agglo_d = AgglomerativeClustering(n_clusters=3)
labels_agglo_d = agglo_d.fit_predict(X_diseases)

# DBSCAN
dbscan_d = DBSCAN(eps=1.5, min_samples=2)
labels_dbscan_d = dbscan_d.fit_predict(X_diseases)

# Adiciona resultados ao DataFrame
df_clusters_d = df.copy()
df_clusters_d['KMeans'] = labels_kmeans_d
df_clusters_d['Agglomerative'] = labels_agglo_d
df_clusters_d['DBSCAN'] = labels_dbscan_d

df_clusters_d.head(30)

In [ ]:
# ================================
# 4. CLUSTERING - ANOS
# ================================

print("\n=== CLUSTERING DE ANOS ===")

years = df.columns

# KMeans
kmeans_y = KMeans(n_clusters=3, random_state=42)
labels_kmeans_y = kmeans_y.fit_predict(X_years)

# Hierarchical
agglo_y = AgglomerativeClustering(n_clusters=3)
labels_agglo_y = agglo_y.fit_predict(X_years)

# DBSCAN
dbscan_y = DBSCAN(eps=1.5, min_samples=2)
labels_dbscan_y = dbscan_y.fit_predict(X_years)

df_years = pd.DataFrame({
    'Year': years,
    'KMeans': labels_kmeans_y,
    'Agglomerative': labels_agglo_y,
    'DBSCAN': labels_dbscan_y
})

print(df_years)


In [ ]:
# ================================
# 5. VISUALIZAÇÃO - HEATMAP
# ================================

plt.figure(figsize=(12, 8))
sns.heatmap(df, cmap='coolwarm')
plt.title("Heatmap - Internações por Doença x Ano")
plt.show()

In [ ]:
# ================================
# 6. DENDROGRAMA (DOENÇAS)
# ================================

linked = linkage(X_diseases, method='ward')

plt.figure(figsize=(12, 6))
dendrogram(linked,
           labels=df.index,
           orientation='right')
plt.title("Dendrograma - Doenças")
plt.show()

In [ ]:
# ================================
# 7. DENDROGRAMA (ANOS)
# ================================

linked_years = linkage(X_years, method='ward')
years_short = [str(year)[-2:] for year in years]

plt.figure(figsize=(10, 5))
dendrogram(linked_years,
           labels=years_short)
plt.title("Dendrograma - Anos")
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# (Opcional, mas recomendado) normalizar os dados
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)

# Converter de volta para DataFrame (mantendo labels)
import pandas as pd
df_scaled = pd.DataFrame(df_scaled, index=df.index, columns=df.columns)

# Clustermap (heatmap + dendrogramas)
sns.clustermap(
    df_scaled,
    cmap='coolwarm',     # 🔴 alto | 🔵 baixo
    method='ward',
    metric='euclidean',
    figsize=(12, 10),
    linewidths=0.1
)

plt.suptitle("Heatmap com Dendrogramas - Internações por Doença x Ano", y=1.02)
plt.show()